# Saved Qwen context-length ablation: 384 vs 768 vs 1,024 tokens
This **does not retrain** Qwen. It reloads the already saved private LoRA adapter/classification head and evaluates the **same seed-42 1,200-row validation subset** three times, changing only tokenizer `max_length`: **384, 768, 1,024**. The same first 128 held-out rows are used for A/B swap stability at each context limit.

Aggregate outputs include multiclass log loss, accuracy, ECE, predicted class counts, confidence, truncation fraction, swap disagreement and runtime. No raw prompts, IDs, row-level probabilities, model weights or competition submission leave the private Kaggle Notebook. Internet stays off. Longer context is an exploratory diagnostic because the adapter itself was trained at 384 tokens.


In [ ]:
from pathlib import Path
import importlib, importlib.metadata, subprocess, sys
from packaging.version import Version
# Keep the previously verified Kaggle compatibility fix before torch/PEFT import.
try:
    torchao_version = importlib.metadata.version('torchao')
except importlib.metadata.PackageNotFoundError:
    torchao_version = None
if torchao_version is not None and Version(torchao_version) <= Version('0.16.0'):
    print('Removing incompatible optional torchao', torchao_version, 'offline')
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
    importlib.invalidate_caches()
    try:
        importlib.metadata.version('torchao')
    except importlib.metadata.PackageNotFoundError:
        pass
    else:
        raise RuntimeError('Incompatible torchao remains installed')
import torch
print('CUDA available:', torch.cuda.is_available(), 'GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
if not torch.cuda.is_available():
    raise RuntimeError('Context ablation requires Kaggle T4 for inference only')
root = Path('/kaggle/input')
if not root.is_dir():
    raise FileNotFoundError('Kaggle input mounts are missing')
mounted = sorted(p.name for p in root.iterdir())
print('Mounted inputs:', mounted)
csv_roots = sorted({p.parent for p in root.rglob('train.csv') if (p.parent / 'test.csv').is_file()})
competition_roots = [p for p in csv_roots if 'llm-classification-finetuning' in str(p).lower()]
if not competition_roots and len(csv_roots) == 1:
    competition_roots = csv_roots
if len(competition_roots) != 1:
    raise FileNotFoundError('Could not identify exactly one official competition input: ' + repr(mounted))
TRAIN = competition_roots[0] / 'train.csv'
base_candidates = [p.parent for p in root.rglob('config.json') if 'qwen2.5' in str(p).lower() and '0.5b' in str(p).lower()]
if len(base_candidates) != 1:
    raise FileNotFoundError('Could not identify exactly one Qwen2.5 0.5B base model input: ' + repr(mounted))
BASE = base_candidates[0]
adapter_candidates = [p.parent for p in root.rglob('adapter_model.safetensors') if (p.parent / 'adapter_config.json').is_file()]
previous = [p for p in adapter_candidates if 'llm-preference-qwen05b-lora-pilot' in str(p)]
if not previous and len(adapter_candidates) == 1:
    previous = adapter_candidates
if len(previous) != 1:
    raise FileNotFoundError('Could not identify exactly one saved prior LoRA adapter: ' + repr(mounted))
ADAPTER = previous[0]
print('Official train:', TRAIN)
print('Base model:', BASE)
print('Saved adapter:', ADAPTER)


In [ ]:
# Generated by python scripts/sync_context_ablation_notebook.py; do not edit this cell by hand.
from pathlib import Path
import sys
source_dir = Path('/kaggle/working/src')
source_dir.mkdir(parents=True, exist_ok=True)
(source_dir / '__init__.py').write_text('', encoding='utf-8')
(source_dir / 'baseline.py').write_text("\"\"\"Leakage-controlled, swap-augmented TF-IDF baseline for Kaggle LLM preference prediction.\n\nThis is a classical ML baseline, not an LLM fine-tuning run.\n\"\"\"\nimport argparse\nimport ast\nimport json\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom scipy.sparse import csr_matrix, hstack, vstack\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import log_loss\nfrom sklearn.model_selection import train_test_split\n\nTARGETS = [\"winner_model_a\", \"winner_model_b\", \"winner_tie\"]\nTEXT_COLUMNS = [\"prompt\", \"response_a\", \"response_b\"]\n\n\ndef flatten_messages(value, max_chars=2400):\n    \"\"\"Normalize Kaggle's serialized lists of turns; cap length for a CPU starter.\"\"\"\n    if value is None or (isinstance(value, float) and np.isnan(value)):\n        return \"\"\n    if isinstance(value, str):\n        text = value.strip()\n        if text.startswith(\"[\"):\n            try:\n                value = json.loads(text)\n            except (ValueError, TypeError):\n                try:\n                    value = ast.literal_eval(text)\n                except (ValueError, SyntaxError):\n                    value = text\n        else:\n            value = text\n    if isinstance(value, (list, tuple)):\n        text = \" \".join(\"\" if item is None else str(item) for item in value)\n    else:\n        text = str(value)\n    return text[:max_chars]\n\n\ndef normalized_frame(df):\n    missing = [c for c in TEXT_COLUMNS if c not in df]\n    if missing:\n        raise ValueError(f\"Missing text columns: {missing}\")\n    return pd.DataFrame(\n        {col: [flatten_messages(v) for v in df[col]] for col in TEXT_COLUMNS},\n        index=df.index,\n    )\n\n\ndef get_labels(df):\n    missing = [c for c in TARGETS if c not in df]\n    if missing:\n        raise ValueError(f\"Missing label columns: {missing}\")\n    y = df[TARGETS].to_numpy(dtype=int)\n    if not np.all(y.sum(axis=1) == 1) or not np.all((y == 0) | (y == 1)):\n        raise ValueError(\"Expected exactly one binary winner label per training row\")\n    return y.argmax(axis=1)\n\n\ndef flip_pairs(df):\n    flipped = df.copy()\n    flipped[\"response_a\"], flipped[\"response_b\"] = (\n        df[\"response_b\"].copy(), df[\"response_a\"].copy()\n    )\n    return flipped\n\n\ndef make_vectorizer(df):\n    # Only fit on training-partition texts; do not fit on held-out validation/test.\n    min_df = 2 if len(df) >= 30 else 1\n    vectorizer = TfidfVectorizer(\n        ngram_range=(1, 2), max_features=35000, min_df=min_df,\n        strip_accents=\"unicode\", sublinear_tf=True, dtype=np.float32,\n    )\n    vectorizer.fit(\n        df[\"prompt\"].tolist() + df[\"response_a\"].tolist() +\n        df[\"response_b\"].tolist()\n    )\n    return vectorizer\n\n\ndef pair_features(df, vectorizer):\n    q = vectorizer.transform(df[\"prompt\"])\n    a = vectorizer.transform(df[\"response_a\"])\n    b = vectorizer.transform(df[\"response_b\"])\n    len_a = df[\"response_a\"].str.len().to_numpy(dtype=np.float32)\n    len_b = df[\"response_b\"].str.len().to_numpy(dtype=np.float32)\n    len_q = df[\"prompt\"].str.len().to_numpy(dtype=np.float32)\n    numeric = np.column_stack([\n        np.log1p(len_a) - np.log1p(len_b),\n        (np.log1p(len_a) + np.log1p(len_b)) / 2,\n        np.log1p(len_q),\n    ]) / 10.0\n    return hstack([q, a - b, (a + b) * 0.5, csr_matrix(numeric)],\n                  format=\"csr\", dtype=np.float32)\n\n\ndef fit_baseline(df, y):\n    vectorizer = make_vectorizer(df)\n    x_original = pair_features(df, vectorizer)\n    x_flipped = pair_features(flip_pairs(df), vectorizer)\n    swapped_labels = np.where(y == 0, 1, np.where(y == 1, 0, 2))\n    model = LogisticRegression(C=2.0, max_iter=300, random_state=42)\n    model.fit(vstack([x_original, x_flipped], format=\"csr\"),\n              np.concatenate([y, swapped_labels]))\n    return {\"vectorizer\": vectorizer, \"model\": model, \"targets\": TARGETS}\n\n\ndef predict_prob(bundle, df):\n    features = pair_features(df, bundle[\"vectorizer\"])\n    raw = bundle[\"model\"].predict_proba(features)\n    out = np.zeros((len(df), len(TARGETS)), dtype=np.float64)\n    for col_idx, class_idx in enumerate(bundle[\"model\"].classes_):\n        out[:, int(class_idx)] = raw[:, col_idx]\n    return out / out.sum(axis=1, keepdims=True)\n\n\ndef train(train_csv, out_dir, validation_fraction=0.15):\n    raw = pd.read_csv(train_csv)\n    df = normalized_frame(raw)\n    y = get_labels(raw)\n    if len(np.unique(y)) != 3 or np.min(np.bincount(y, minlength=3)) < 2:\n        raise ValueError(\"Each class must have at least two examples for validation\")\n    x_tr, x_val, y_tr, y_val = train_test_split(\n        df, y, test_size=validation_fraction, random_state=42, stratify=y\n    )\n    validation_bundle = fit_baseline(x_tr, y_tr)\n    val_probs = predict_prob(validation_bundle, x_val)\n    metrics = {\n        \"validation_log_loss\": float(log_loss(y_val, val_probs, labels=[0, 1, 2])),\n        \"train_rows\": int(len(x_tr)),\n        \"validation_rows\": int(len(x_val)),\n        \"full_rows\": int(len(df)),\n        \"seed\": 42,\n        \"note\": \"Random stratified split; not a competition leaderboard result.\",\n    }\n    output = Path(out_dir)\n    output.mkdir(parents=True, exist_ok=True)\n    (output / \"validation_metrics.json\").write_text(\n        json.dumps(metrics, indent=2) + \"\\n\", encoding=\"utf-8\"\n    )\n    # Refit using all labeled data only after holding out validation above.\n    joblib.dump(fit_baseline(df, y), output / \"baseline.joblib\")\n    return metrics\n\n\ndef predict(test_csv, model_path, out_csv):\n    test = pd.read_csv(test_csv)\n    if \"id\" not in test.columns:\n        raise ValueError(\"Test CSV must contain id\")\n    bundle = joblib.load(model_path)  # Load only artifacts you created/trust.\n    probabilities = predict_prob(bundle, normalized_frame(test))\n    result = pd.DataFrame(probabilities, columns=TARGETS)\n    result.insert(0, \"id\", test[\"id\"])\n    Path(out_csv).parent.mkdir(parents=True, exist_ok=True)\n    result.to_csv(out_csv, index=False)\n    return result\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    sub = parser.add_subparsers(dest=\"command\", required=True)\n    fit = sub.add_parser(\"train\")\n    fit.add_argument(\"--train\", default=\"data/train.csv\")\n    fit.add_argument(\"--out\", default=\"artifacts\")\n    infer = sub.add_parser(\"predict\")\n    infer.add_argument(\"--test\", default=\"data/test.csv\")\n    infer.add_argument(\"--model\", default=\"artifacts/baseline.joblib\")\n    infer.add_argument(\"--out\", default=\"submission.csv\")\n    args = parser.parse_args()\n    if args.command == \"train\":\n        print(json.dumps(train(args.train, args.out), indent=2))\n    else:\n        print(f\"Wrote {len(predict(args.test, args.model, args.out))} rows: {args.out}\")\n\n\nif __name__ == \"__main__\":\n    main()\n", encoding='utf-8')
(source_dir / 'finetune_lora.py').write_text("\"\"\"GPU pilot: fine-tune an offline Qwen2.5-0.5B base sequence classifier with LoRA.\n\nOnly run after attaching legitimately accessible model weights and official Kaggle\ncompetition data. This is a pilot; no actual GPU experiment is claimed here.\n\nWith offline Kaggle submissions, attach the *complete* base model directory\n(config, tokenizer and weights) as an Input; never fetch from Hugging Face at runtime.\n\"\"\"\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom scipy.special import softmax\nfrom sklearn.metrics import log_loss\nfrom sklearn.model_selection import train_test_split\n\nfrom src.baseline import TARGETS, flatten_messages, flip_pairs, get_labels, normalized_frame\nfrom src.length_baseline import fit_length_model, predict_length_model\n\n\ndef render_pair(row):\n    \"\"\"Fixed prompt template and truncation; do not include train-only model names.\"\"\"\n    question = flatten_messages(row[\"prompt\"], max_chars=1200)\n    a = flatten_messages(row[\"response_a\"], max_chars=2400)\n    b = flatten_messages(row[\"response_b\"], max_chars=2400)\n    return (\n        \"A human gave two chatbots the same user request.\\n\"\n        f\"User request: {question}\\n\"\n        f\"Response A: {a}\\n\"\n        f\"Response B: {b}\\n\"\n        \"Predict whether the human prefers response A, response B, or a tie.\"\n    )\n\n\ndef swap_labels(labels):\n    labels = np.asarray(labels, dtype=np.int64)\n    if not np.isin(labels, [0, 1, 2]).all():\n        raise ValueError(\"Expected A=0, B=1, tie=2\")\n    return np.where(labels == 0, 1, np.where(labels == 1, 0, 2))\n\n\ndef train_and_predict(args):\n    # Lazy imports ensure that unit tests do not need GPU-only dependencies.\n    import torch\n    from peft import LoraConfig, TaskType, get_peft_model\n    from torch.utils.data import Dataset\n    from transformers import (\n        AutoModelForSequenceClassification, AutoTokenizer,\n        DataCollatorWithPadding, Trainer, TrainingArguments,\n    )\n\n    if not torch.cuda.is_available():\n        raise RuntimeError(\"This LoRA pilot requires a CUDA GPU; use the CPU baseline otherwise.\")\n    model_dir = Path(args.base_model).expanduser()\n    if not (model_dir / \"config.json\").exists():\n        raise FileNotFoundError(\n            \"Supply the complete offline base model directory via --base-model; \"\n            f\"no config.json found in {model_dir}\"\n        )\n    if args.pilot_rows != 0 and args.pilot_rows < 30:\n        raise ValueError(\"pilot_rows must be 0 (all rows) or at least 30\")\n\n    torch.manual_seed(args.seed)\n    np.random.seed(args.seed)\n    raw = pd.read_csv(args.train)\n    df = normalized_frame(raw)\n    y = get_labels(raw)\n    if np.min(np.bincount(y, minlength=3)) < 2:\n        raise ValueError(\"Each class needs at least two rows\")\n    x_train, x_val, y_train, y_val = train_test_split(\n        df, y, test_size=0.15, stratify=y, random_state=args.seed\n    )\n    # Select an optional, stratified validation subset *before* any GPU training.\n    # The complete official split remains unmodified in the source data.\n    max_validation_rows = getattr(args, \"max_validation_rows\", 0)\n    if max_validation_rows and len(x_val) > max_validation_rows:\n        x_val, _, y_val, _ = train_test_split(\n            x_val, y_val, train_size=max_validation_rows,\n            stratify=y_val, random_state=args.seed\n        )\n    # Cap *training only*. Keep validation untouched by augmentation.\n    if args.pilot_rows and len(x_train) > args.pilot_rows:\n        x_train, _, y_train, _ = train_test_split(\n            x_train, y_train, train_size=args.pilot_rows,\n            stratify=y_train, random_state=args.seed\n        )\n    x_original, y_original = x_train.copy(), y_train.copy()\n    # A fair, matched, same-training-size reference for the GPU pilot.\n    length_reference = fit_length_model(x_original, y_original, c=10.0)\n    length_reference_prob = predict_length_model(length_reference, x_val)\n    matched_length_loss = float(log_loss(y_val, length_reference_prob, labels=[0, 1, 2]))\n    if args.swap_train:\n        x_train = pd.concat(\n            [x_original, flip_pairs(x_original)], ignore_index=True\n        )\n        y_train = np.concatenate([y_original, swap_labels(y_original)])\n\n    tokenizer = AutoTokenizer.from_pretrained(\n        model_dir, local_files_only=True, trust_remote_code=False\n    )\n    if tokenizer.pad_token_id is None:\n        if tokenizer.eos_token is None:\n            raise ValueError(\"Tokenizer has neither pad nor EOS token\")\n        tokenizer.pad_token = tokenizer.eos_token\n\n    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\n    model = AutoModelForSequenceClassification.from_pretrained(\n        model_dir, num_labels=3, torch_dtype=dtype,\n        local_files_only=True, trust_remote_code=False,\n    )\n    model.config.pad_token_id = tokenizer.pad_token_id\n    model.config.use_cache = False\n    model = get_peft_model(\n        model,\n        LoraConfig(\n            task_type=TaskType.SEQ_CLS,\n            r=8, lora_alpha=16, lora_dropout=0.05,\n            target_modules=[\"q_proj\", \"v_proj\"],\n            modules_to_save=[\"score\"],\n        ),\n    )\n\n    class PairDataset(Dataset):\n        def __init__(self, frame, labels=None):\n            self.texts = [render_pair(row) for row in frame.to_dict(\"records\")]\n            self.labels = labels\n\n        def __len__(self):\n            return len(self.texts)\n\n        def __getitem__(self, i):\n            encoded = tokenizer(\n                self.texts[i], truncation=True, max_length=args.max_length\n            )\n            if self.labels is not None:\n                encoded[\"labels\"] = int(self.labels[i])\n            return encoded\n\n    output = Path(args.output)\n    output.mkdir(parents=True, exist_ok=True)\n    config = TrainingArguments(\n        output_dir=str(output / \"trainer\"),\n        num_train_epochs=args.epochs,\n        per_device_train_batch_size=args.batch_size,\n        per_device_eval_batch_size=args.eval_batch_size,\n        gradient_accumulation_steps=args.grad_accum,\n        learning_rate=args.learning_rate,\n        weight_decay=0.01,\n        lr_scheduler_type=\"cosine\",\n        warmup_ratio=0.05,\n        fp16=dtype == torch.float16,\n        bf16=dtype == torch.bfloat16,\n        eval_strategy=\"no\",\n        save_strategy=\"no\",\n        logging_strategy=\"steps\",\n        logging_steps=25,\n        report_to=\"none\",\n        remove_unused_columns=False,\n        dataloader_num_workers=0,\n        seed=args.seed,\n    )\n    trainer = Trainer(\n        model=model,\n        args=config,\n        train_dataset=PairDataset(x_train, y_train),\n        data_collator=DataCollatorWithPadding(\n            tokenizer=tokenizer, pad_to_multiple_of=8\n        ),\n        processing_class=tokenizer,\n    )\n    trainer.train()\n    val_logits = trainer.predict(PairDataset(x_val)).predictions\n    if isinstance(val_logits, tuple):\n        val_logits = val_logits[0]\n    val_prob = softmax(np.asarray(val_logits, dtype=np.float64), axis=-1)\n    metrics = {\n        \"validation_log_loss\": float(log_loss(y_val, val_prob, labels=[0, 1, 2])),\n        \"validation_rows\": len(x_val),\n        \"train_rows_after_augmentation\": len(x_train),\n        \"pilot_rows\": args.pilot_rows,\n        \"seed\": args.seed,\n        \"max_length_tokens\": args.max_length,\n        \"base_model_dir\": model_dir.name,\n        \"training_type\": \"Qwen2.5-0.5B base sequence classification head + LoRA\",\n        \"matched_length_reference_log_loss\": matched_length_loss,\n        \"reference_note\": \"Length-only C=10 refit on exactly the GPU pilot original training subset; identical held-out validation rows.\",\n        \"caution\": \"Preliminary pilot; independent replication and full-data run pending.\",\n    }\n    # Probe original/swap consistency on a bounded held-out subset.\n    subset = x_val.iloc[: min(128, len(x_val))]\n    original_logits = trainer.predict(PairDataset(subset)).predictions\n    swapped_logits = trainer.predict(PairDataset(flip_pairs(subset))).predictions\n    if isinstance(original_logits, tuple):\n        original_logits = original_logits[0]\n    if isinstance(swapped_logits, tuple):\n        swapped_logits = swapped_logits[0]\n    original_prob = softmax(np.asarray(original_logits, dtype=np.float64), axis=-1)\n    swapped_prob = softmax(np.asarray(swapped_logits, dtype=np.float64), axis=-1)[:, [1, 0, 2]]\n    metrics[\"swap_probe_mean_abs_difference\"] = float(\n        np.abs(original_prob - swapped_prob).mean()\n    )\n    # Preserve successful validation metrics even if adapter saving or optional\n    # 25K-row Kaggle test inference fails. The JSON is aggregate-only.\n    metrics[\"pipeline_status\"] = \"validation_completed\"\n    try:\n        adapter_dir = output / \"adapter\"\n        trainer.model.save_pretrained(adapter_dir)\n        tokenizer.save_pretrained(adapter_dir)\n        metrics[\"pipeline_status\"] = \"adapter_saved\"\n\n        if args.test:\n            test = pd.read_csv(args.test)\n            if \"id\" not in test.columns:\n                raise ValueError(\"Test data require id column\")\n            test_frame = normalized_frame(test)\n            test_logits = trainer.predict(PairDataset(test_frame)).predictions\n            if isinstance(test_logits, tuple):\n                test_logits = test_logits[0]\n            probs = softmax(np.asarray(test_logits, dtype=np.float64), axis=-1)\n            submission = pd.DataFrame(probs, columns=TARGETS)\n            submission.insert(0, \"id\", test[\"id\"])\n            submission_path = Path(args.submission)\n            submission_path.parent.mkdir(parents=True, exist_ok=True)\n            submission.to_csv(submission_path, index=False)\n            metrics[\"submission_rows\"] = int(len(submission))\n            metrics[\"pipeline_status\"] = \"submission_completed\"\n            print(f\"Submission written to {submission_path}\")\n    except Exception as exc:\n        metrics[\"pipeline_status\"] = \"downstream_failed\"\n        metrics[\"downstream_error_type\"] = type(exc).__name__\n        raise\n    finally:\n        (output / \"gpu_pilot_metrics.json\").write_text(\n            json.dumps(metrics, indent=2) + \"\\n\", encoding=\"utf-8\"\n        )\n    print(json.dumps(metrics, indent=2))\n    return metrics\n\n\ndef parse_args():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\"--train\", default=\"data/train.csv\")\n    p.add_argument(\"--test\", default=None)\n    p.add_argument(\"--base-model\", required=True,\n                   help=\"Complete *local* Qwen2.5-0.5B weights/tokenizer folder\")\n    p.add_argument(\"--output\", default=\"artifacts/gpu_pilot\")\n    p.add_argument(\"--submission\", default=\"submission.csv\")\n    p.add_argument(\"--pilot-rows\", type=int, default=4000,\n                   help=\"Training cap excluding held-out validation; 0 uses all train rows\")\n    p.add_argument(\"--max-length\", type=int, default=384)\n    p.add_argument(\"--max-validation-rows\", type=int, default=1200,\n                   help=\"Stratified subset of held-out validation for pilot runtime; 0 uses all\")\n    p.add_argument(\"--epochs\", type=float, default=1.0)\n    p.add_argument(\"--batch-size\", type=int, default=2)\n    p.add_argument(\"--eval-batch-size\", type=int, default=4)\n    p.add_argument(\"--grad-accum\", type=int, default=8)\n    p.add_argument(\"--learning-rate\", type=float, default=2e-4)\n    p.add_argument(\"--seed\", type=int, default=42)\n    p.add_argument(\"--no-swap-train\", action=\"store_false\", dest=\"swap_train\")\n    p.set_defaults(swap_train=True)\n    return p.parse_args()\n\n\nif __name__ == \"__main__\":\n    train_and_predict(parse_args())\n", encoding='utf-8')
(source_dir / 'length_baseline.py').write_text("\"\"\"Nested-tuned, length-only A/B/tie baseline, exploratory validation.\n\nThis tests whether simple observable character lengths provide predictive signal.\nIt neither measures warmth nor establishes a causal preference for verbosity.\nOnly aggregate JSON may be published; official Kaggle data stay local.\n\"\"\"\nimport argparse\nimport hashlib\nimport json\nimport os\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport sklearn\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import log_loss\nfrom sklearn.model_selection import train_test_split\n\nfrom src.baseline import TARGETS, flatten_messages, flip_pairs, get_labels\n\n\ndef file_sha256(path):\n    digest = hashlib.sha256()\n    with open(path, 'rb') as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b''):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef feature_matrix(df):\n    \"\"\"No semantic word features: lengths of complete joined conversation fields.\"\"\"\n    def lengths(col):\n        return np.asarray(\n            [len(flatten_messages(value, max_chars=2_000_000)) for value in df[col]],\n            dtype=np.float64,\n        )\n    a = np.log1p(lengths(\"response_a\"))\n    b = np.log1p(lengths(\"response_b\"))\n    q = np.log1p(lengths(\"prompt\"))\n    delta = a - b\n    return np.column_stack([\n        delta,\n        np.abs(delta),\n        (a + b) / 2,\n        q,\n        delta * (q / 10),\n        (a - b) / np.maximum((a + b), 1.0),\n    ])\n\n\ndef fit_length_model(frame, labels, c=1.0):\n    original = feature_matrix(frame)\n    swapped = feature_matrix(flip_pairs(frame))\n    swapped_labels = np.where(labels == 0, 1, np.where(labels == 1, 0, 2))\n    model = LogisticRegression(C=c, max_iter=700, random_state=42)\n    model.fit(\n        np.vstack([original, swapped]),\n        np.concatenate([labels, swapped_labels])\n    )\n    return model\n\n\ndef predict_length_model(model, frame):\n    raw = model.predict_proba(feature_matrix(frame))\n    out = np.zeros((len(frame), 3), dtype=np.float64)\n    for idx, label in enumerate(model.classes_):\n        out[:, int(label)] = raw[:, idx]\n    return out\n\n\ndef logloss_delta_interval(labels, prediction, reference, draws=1000, seed=42):\n    \"\"\"Bootstrap per-row excess log loss; negative means model lower loss.\"\"\"\n    y = np.asarray(labels, dtype=np.int64)\n    selected_model = np.clip(prediction[np.arange(len(y)), y], 1e-15, 1)\n    selected_reference = np.clip(reference[np.arange(len(y)), y], 1e-15, 1)\n    delta = -np.log(selected_model) + np.log(selected_reference)\n    rng = np.random.default_rng(seed)\n    draws_arr = np.array([\n        delta[rng.integers(0, len(delta), len(delta))].mean()\n        for _ in range(draws)\n    ])\n    return [float(x) for x in np.quantile(draws_arr, [0.025, 0.975])]\n\n\ndef run_pilot(train_csv, out_dir, sample_size=12000, seed=42):\n    source = Path(train_csv)\n    output = Path(out_dir)\n    output.mkdir(parents=True, exist_ok=True)\n    raw = pd.read_csv(source)\n    labels = get_labels(raw)\n    if len(raw) < 100 or np.min(np.bincount(labels, minlength=3)) < 10:\n        raise ValueError(\"Need >=100 labeled examples and >=10 per class\")\n    if sample_size < 0 or (sample_size > 0 and sample_size < 100):\n        raise ValueError(\"Pilot size should be 0 for all rows or >=100\")\n    if sample_size and sample_size < len(raw):\n        chosen, _ = train_test_split(\n            np.arange(len(raw)), train_size=sample_size,\n            stratify=labels, random_state=seed,\n        )\n        pilot = raw.iloc[chosen].reset_index(drop=True)\n    else:\n        pilot = raw.reset_index(drop=True)\n    pilot_y = get_labels(pilot)\n    if np.min(np.bincount(pilot_y, minlength=3)) < 10:\n        raise ValueError(\n            'Sampled pilot must contain at least 10 rows per class for nested splitting'\n        )\n    # Exactly the outer split used in the 2026-09-23 TF-IDF benchmark.\n    outer_train, outer_val, y_train, y_val = train_test_split(\n        pilot, pilot_y, test_size=0.15, random_state=42, stratify=pilot_y\n    )\n    # Hyperparameter selection must be based only on an INNER calibration fold.\n    inner_train, inner_val, inner_y, inner_val_y = train_test_split(\n        outer_train, y_train, test_size=0.20,\n        random_state=1337, stratify=y_train\n    )\n    candidate_c = [0.01, 0.1, 1.0, 10.0]\n    inner_results = {}\n    for c in candidate_c:\n        model = fit_length_model(inner_train, inner_y, c=c)\n        inner_results[str(c)] = float(\n            log_loss(inner_val_y, predict_length_model(model, inner_val),\n                     labels=[0, 1, 2])\n        )\n    best_c = min(candidate_c, key=lambda c: inner_results[str(c)])\n    model = fit_length_model(outer_train, y_train, c=best_c)\n    prediction = predict_length_model(model, outer_val)\n    uniform = np.full_like(prediction, 1 / 3)\n    prior = np.bincount(y_train, minlength=3).astype(np.float64)\n    prior /= prior.sum()\n    prior_probs = np.tile(prior, (len(y_val), 1))\n    metrics = {\n        \"source\":\"Official Kaggle LLM Classification Finetuning training CSV\",\n        \"source_file_sha256\":file_sha256(source),\n        \"official_training_rows\":int(len(raw)),\n        \"pilot_rows\":int(len(pilot)),\n        \"outer_validation_rows\":int(len(y_val)),\n        \"outer_validation_log_loss_length_only\":float(log_loss(y_val, prediction,labels=[0,1,2])),\n        \"outer_validation_log_loss_training_prior\":float(log_loss(y_val, prior_probs,labels=[0,1,2])),\n        \"outer_validation_log_loss_uniform\":float(log_loss(y_val, uniform,labels=[0,1,2])),\n        \"length_minus_prior_log_loss_bootstrap_95pct\":logloss_delta_interval(\n            y_val, prediction, prior_probs, seed=seed,\n        ),\n        \"inner_validation_log_loss_by_c\":inner_results,\n        \"selected_c\":best_c,\n        \"outer_seed\":42,\n        \"inner_seed\":1337,\n        \"pilot_seed\":seed,\n        \"note\":\"EXPLORATORY comparison. Outer fold overlaps initial TF-IDF benchmark; do not reuse it indefinitely for model selection. Not a Kaggle submission.\",\n        \"constraints\":[\"Row-random splitting; repeated prompts may cross splits.\",\"Character lengths are not measures of style or warmth.\",\"Selected C tuned on inner fold only.\",\"No text features: performance reflects length correlates, not causal effects.\"],\n        \"environment\":{\n            \"scikit_learn\":sklearn.__version__,\n            \"pandas\":pd.__version__,\n            \"github_sha\":os.getenv(\"GITHUB_SHA\",\"local\"),\n            \"run_utc\":datetime.now(timezone.utc).isoformat(),\n        }\n    }\n    (output/\"summary.json\").write_text(\n        json.dumps(metrics,indent=2)+\"\\n\",encoding=\"utf-8\"\n    )\n    print(\"Official training rows:\",metrics[\"official_training_rows\"])\n    print(\"Pilot training rows:\",metrics[\"pilot_rows\"])\n    print(\"Chosen inner-fold regularization C:\",best_c)\n    print(\"Outer length-only log loss:\",metrics[\"outer_validation_log_loss_length_only\"])\n    print(\"Outer train-prior log loss:\",metrics[\"outer_validation_log_loss_training_prior\"])\n    print(\"95% bootstrap (length minus prior):\",metrics[\"length_minus_prior_log_loss_bootstrap_95pct\"])\n    print(\"Aggregate summary saved.\")\n    return metrics\n\n\nif __name__ == \"__main__\":\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\"--train\",default=\"data/train.csv\")\n    p.add_argument(\"--out\",default=\"artifacts/official_length_pilot\")\n    p.add_argument(\"--sample-size\",type=int,default=12000)\n    a=p.parse_args()\n    run_pilot(a.train,a.out,a.sample_size)\n", encoding='utf-8')
(source_dir / 'saved_adapter_inference.py').write_text("\"\"\"Inference-only diagnosis of an EXISTING Kaggle Qwen-LoRA adapter.\n\nNo model training, raw-row export, checkpoint publishing or competition\nsubmission. Only this module's aggregate JSON is suitable for GitHub artifacts.\nThe original 2026-09-24 GPU pilot used seed 42, a 15% row-stratified outer\nsplit and a stratified 1,200-row subsample of that held-out fold.\n\"\"\"\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom scipy.special import softmax\nfrom sklearn.metrics import confusion_matrix, log_loss\nfrom sklearn.model_selection import train_test_split\n\nfrom src.baseline import TARGETS, flatten_messages, flip_pairs, get_labels, normalized_frame\nfrom src.finetune_lora import render_pair\n\n\ndef exact_pilot_validation(raw, seed=42, max_validation_rows=1200):\n    \"\"\"Reproduce *both* split calls used by the successful Kaggle pilot.\"\"\"\n    frame = normalized_frame(raw)\n    labels = get_labels(raw)\n    _, val_frame, _, val_labels = train_test_split(\n        frame, labels, test_size=0.15, random_state=seed, stratify=labels\n    )\n    if max_validation_rows and len(val_frame) > max_validation_rows:\n        val_frame, _, val_labels, _ = train_test_split(\n            val_frame, val_labels, train_size=max_validation_rows,\n            random_state=seed, stratify=val_labels\n        )\n    return val_frame, np.asarray(val_labels, dtype=int)\n\n\ndef validated_swap_rows(requested, available):\n    \"\"\"Reject empty swap probes before touching expensive model inference.\"\"\"\n    if requested <= 0 or available <= 0:\n        raise ValueError(\"--swap-rows and available validation rows must be positive\")\n    return min(requested, available)\n\n\ndef predict_in_batches(model, tokenizer, frame, device, max_length=384, batch_size=8):\n    \"\"\"Use saved adapter in eval/inference mode only; return in-memory probabilities.\"\"\"\n    import torch\n    if batch_size <= 0:\n        raise ValueError(\"batch_size must be positive\")\n    if len(frame) == 0:\n        raise ValueError(\"Inference batch frame must not be empty\")\n    result = []\n    model.eval()\n    with torch.inference_mode():\n        for start in range(0, len(frame), batch_size):\n            chunk = frame.iloc[start:start + batch_size]\n            rendered = [render_pair(row) for row in chunk.to_dict(\"records\")]\n            enc = tokenizer(\n                rendered, truncation=True, max_length=max_length,\n                padding=True, return_tensors=\"pt\"\n            )\n            batch = {key: value.to(device) for key, value in enc.items()}\n            logits = model(**batch).logits.detach().float().cpu().numpy()\n            if logits.ndim != 2 or logits.shape[1] != 3:\n                raise ValueError(f\"Expected exactly 3 logits, received {logits.shape}\")\n            result.append(softmax(logits.astype(np.float64), axis=1))\n    return np.vstack(result)\n\n\ndef token_truncation_report(tokenizer, frame, max_length=384):\n    \"\"\"Count truncated rendered text locally; never include any text in the report.\"\"\"\n    lengths = np.array([\n        len(tokenizer(render_pair(row), add_special_tokens=True, truncation=False)[\"input_ids\"])\n        for row in frame.to_dict(\"records\")\n    ], dtype=int)\n    return {\n        \"evaluated_rows\": int(len(lengths)),\n        \"sequence_max_tokens\": max_length,\n        \"rendered_token_length_mean\": float(lengths.mean()),\n        \"rendered_token_length_median\": float(np.median(lengths)),\n        \"rendered_token_length_p90\": float(np.percentile(lengths, 90)),\n        \"rendered_token_length_max\": int(lengths.max()),\n        \"fraction_rendered_sequence_exceeds_token_limit\":\n            float(np.mean(lengths > max_length)),\n        \"warning\": \"Also includes earlier character caps in normalized_frame (2400 per column), \"\n                   \"render_pair (1200 prompt, 2400 per response); full original content \"\n                   \"beyond those caps is not included in token-length counts.\",\n    }\n\n\ndef character_cap_report(raw_rows):\n    \"\"\"Measure the raw character caps applied before the tokenizer.\"\"\"\n    caps = {\"prompt\": 1200, \"response_a\": 2400, \"response_b\": 2400}\n    results = {}\n    for field, cap in caps.items():\n        lengths = np.asarray([\n            len(flatten_messages(v, max_chars=10_000_000))\n            for v in raw_rows[field]\n        ])\n        results[field] = {\n            \"cap_characters\": cap,\n            \"fraction_raw_fields_exceed_cap\": float(np.mean(lengths > cap)),\n        }\n    return results\n\n\ndef aggregate_predictions(y, probabilities, swapped_aligned=None):\n    \"\"\"Compute only class-wise and overall diagnostics, no row-level exports.\"\"\"\n    y = np.asarray(y, dtype=int)\n    p = np.asarray(probabilities, dtype=np.float64)\n    if p.shape != (len(y), 3) or not np.isfinite(p).all():\n        raise ValueError(\"Expected finite Nx3 probabilities\")\n    if np.any(p < -1e-7) or not np.allclose(p.sum(axis=1), 1, atol=1e-5):\n        raise ValueError(\"Invalid class probabilities\")\n    if not np.isin(y, [0, 1, 2]).all() or not len(y):\n        raise ValueError(\"Expected nonempty validation labels 0,1,2\")\n    p = np.clip(p, 1e-12, 1.0)\n    p /= p.sum(axis=1, keepdims=True)\n    predicted = np.argmax(p, axis=1)\n    counts = np.bincount(predicted, minlength=3)\n    confusion = confusion_matrix(y, predicted, labels=[0, 1, 2])\n    confidence = p.max(axis=1)\n    correct = (predicted == y).astype(float)\n    entropy = -(p * np.log(p)).sum(axis=1)\n    raw_bins = np.minimum((confidence * 10).astype(int), 9)\n    calibration = []\n    for k in range(10):\n        mask = raw_bins == k\n        calibration.append({\n            \"lower_confidence\": k / 10,\n            \"upper_confidence\": (k + 1) / 10,\n            \"n\": int(mask.sum()),\n            \"mean_confidence\": float(confidence[mask].mean()) if mask.any() else None,\n            \"accuracy\": float(correct[mask].mean()) if mask.any() else None,\n        })\n    ece = sum(\n        (row[\"n\"] / len(y)) * abs(row[\"mean_confidence\"] - row[\"accuracy\"])\n        for row in calibration if row[\"n\"]\n    )\n    classwise = {}\n    for idx, name in enumerate(TARGETS):\n        mask = y == idx\n        classwise[name] = {\n            \"true_n\": int(mask.sum()),\n            \"predicted_n\": int(counts[idx]),\n            \"mean_true_class_probability\":\n                float(p[mask, idx].mean()) if mask.any() else None,\n            \"mean_negative_log_likelihood\":\n                float(-np.log(p[mask, idx]).mean()) if mask.any() else None,\n            \"recall\": float((predicted[mask] == idx).mean()) if mask.any() else None,\n        }\n    output = {\n        \"n\": int(len(y)),\n        \"multiclass_log_loss\": float(log_loss(y, p, labels=[0, 1, 2])),\n        \"accuracy\": float(correct.mean()),\n        \"multiclass_brier_score\": float(np.mean(np.sum(\n            (p - np.eye(3)[y]) ** 2, axis=1\n        ))),\n        \"mean_prediction_entropy_nats\": float(entropy.mean()),\n        \"mean_max_probability\": float(confidence.mean()),\n        \"median_max_probability\": float(np.median(confidence)),\n        \"p90_max_probability\": float(np.percentile(confidence, 90)),\n        \"predicted_counts\": {\n            name: int(counts[i]) for i, name in enumerate(TARGETS)\n        },\n        \"mean_predicted_probabilities\": {\n            name: float(p[:, i].mean()) for i, name in enumerate(TARGETS)\n        },\n        \"true_counts\": {\n            name: int((y == i).sum()) for i, name in enumerate(TARGETS)\n        },\n        \"confusion_matrix_true_rows_predicted_columns\": confusion.astype(int).tolist(),\n        \"classwise\": classwise,\n        \"confidence_reliability_bins\": calibration,\n        \"expected_calibration_error_10_bin\": float(ece),\n        \"notes\": [\n            \"ECE uses the maximum predicted probability and ten equal-width bins. \"\n            \"Small-bin estimates are noisy.\",\n            \"Scores are for the original Qwen pilot row-random validation subset, \"\n            \"not the prompt-grouped reporting holdout or Kaggle leaderboard.\",\n        ],\n    }\n    if swapped_aligned is not None:\n        aligned = np.asarray(swapped_aligned, dtype=np.float64)\n        if aligned.shape != p.shape or not np.isfinite(aligned).all():\n            raise ValueError(\"Swapped probabilities must match original shape\")\n        err = np.abs(p - aligned)\n        output[\"swap_diagnostic\"] = {\n            \"n\": int(len(y)),\n            \"mean_absolute_probability_disagreement\": float(err.mean()),\n            \"fraction_rows_with_any_class_difference_above_0p2\":\n                float(np.mean(err.max(axis=1) > 0.2)),\n            \"fraction_rows_with_changed_predicted_winner\":\n                float(np.mean(predicted != np.argmax(aligned, axis=1))),\n            \"meaning\": \"Model order instability after relabeling swapped A/B. \"\n                       \"Not a measurement of human presentation bias.\",\n        }\n    return output\n\n\ndef main(args):\n    # Model loading is intentionally lazy; synthetic tests run entirely on CPU.\n    import torch\n    from peft import PeftModel\n    from safetensors import safe_open\n    from transformers import AutoModelForSequenceClassification, AutoTokenizer\n\n    base_dir = Path(args.base_model)\n    adapter_dir = Path(args.adapter)\n    if not (base_dir / \"config.json\").is_file():\n        raise FileNotFoundError(\"Missing local pretrained base model config\")\n    if not (adapter_dir / \"adapter_config.json\").is_file():\n        raise FileNotFoundError(\"Missing saved adapter configuration\")\n    if not (adapter_dir / \"adapter_model.safetensors\").is_file():\n        raise FileNotFoundError(\"Missing saved adapter weights\")\n    with safe_open(adapter_dir / \"adapter_model.safetensors\", framework=\"pt\",\n                   device=\"cpu\") as handle:\n        keys = list(handle.keys())\n    head_keys = [key for key in keys if \"score\" in key and key.endswith(\".weight\")]\n    if not head_keys:\n        raise ValueError(\"Saved adapter has no classifier score weight; \"\n                         \"cannot claim original head was restored\")\n    if not torch.cuda.is_available():\n        raise RuntimeError(\"Inference-only Notebook requires Kaggle CUDA accelerator\")\n    raw = pd.read_csv(args.train)\n    x_val, y_val = exact_pilot_validation(raw, seed=args.seed,\n                                         max_validation_rows=args.max_validation_rows)\n    # Map back only to these original raw rows for character-cap counts.\n    raw_val = raw.loc[x_val.index]\n    n_swap = validated_swap_rows(args.swap_rows, len(x_val))\n    tokenizer = AutoTokenizer.from_pretrained(base_dir, local_files_only=True,\n                                              trust_remote_code=False)\n    if tokenizer.pad_token_id is None:\n        if tokenizer.eos_token is None:\n            raise ValueError(\"Tokenizer has neither pad nor EOS\")\n        tokenizer.pad_token = tokenizer.eos_token\n    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\n    base = AutoModelForSequenceClassification.from_pretrained(\n        base_dir, num_labels=3, torch_dtype=dtype,\n        local_files_only=True, trust_remote_code=False\n    )\n    base.config.pad_token_id = tokenizer.pad_token_id\n    base.config.use_cache = False\n    model = PeftModel.from_pretrained(\n        base, adapter_dir, is_trainable=False, local_files_only=True\n    )\n    model.to(\"cuda:0\")\n    model.eval()\n\n    probs = predict_in_batches(model, tokenizer, x_val, device=\"cuda:0\",\n                               max_length=args.max_length, batch_size=args.batch_size)\n    swapped = predict_in_batches(\n        model, tokenizer, flip_pairs(x_val.iloc[:n_swap]), \"cuda:0\",\n        max_length=args.max_length, batch_size=args.batch_size\n    )[:, [1, 0, 2]]\n    aggregated = aggregate_predictions(y_val, probs)\n    swap_aggregated = aggregate_predictions(\n        y_val[:n_swap], probs[:n_swap], swapped_aligned=swapped\n    )[\"swap_diagnostic\"]\n    aggregated[\"swap_diagnostic\"] = swap_aggregated\n    aggregated[\"rendered_token_truncation\"] = token_truncation_report(\n        tokenizer, x_val, max_length=args.max_length\n    )\n    aggregated[\"original_character_cap_rates\"] = character_cap_report(raw_val)\n    aggregated[\"model_check\"] = {\n        \"loaded_saved_classifier_weight_tensors\": len(head_keys),\n        \"num_classes\": 3,\n        \"adapter_source\": \"previous private Kaggle Qwen LoRA v2 output\",\n        \"no_new_training\": True,\n    }\n    prior = Path(args.previous_metrics)\n    if prior.is_file():\n        old = json.loads(prior.read_text(encoding=\"utf-8\"))\n        old_loss = old.get(\"validation_log_loss\")\n        old_swap = old.get(\"swap_probe_mean_abs_difference\")\n        aggregated[\"reproducibility\"] = {\n            \"previous_validation_log_loss\": old_loss,\n            \"difference_in_log_loss\":\n                aggregated[\"multiclass_log_loss\"] - float(old_loss)\n                if old_loss is not None else None,\n            \"previous_swap_mean_abs_disagreement\": old_swap,\n            \"difference_in_swap_mean_abs_disagreement\":\n                aggregated[\"swap_diagnostic\"][\"mean_absolute_probability_disagreement\"] -\n                float(old_swap) if old_swap is not None else None,\n        }\n    Path(args.output).parent.mkdir(parents=True, exist_ok=True)\n    Path(args.output).write_text(\n        json.dumps(aggregated, ensure_ascii=False, indent=2) + \"\\n\",\n        encoding=\"utf-8\"\n    )\n    # No raw prompts, text, IDs or per-row probabilities are written to disk.\n    print(json.dumps({\n        \"validation_rows\": aggregated[\"n\"],\n        \"log_loss\": aggregated[\"multiclass_log_loss\"],\n        \"predicted_counts\": aggregated[\"predicted_counts\"],\n        \"mean_confidence\": aggregated[\"mean_max_probability\"],\n        \"swap_mean_absolute_disagreement\":\n            aggregated[\"swap_diagnostic\"][\"mean_absolute_probability_disagreement\"],\n        \"token_truncation_fraction\":\n            aggregated[\"rendered_token_truncation\"][\"fraction_rendered_sequence_exceeds_token_limit\"],\n    }, indent=2))\n    return aggregated\n\n\nif __name__ == \"__main__\":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\"--train\", required=True)\n    parser.add_argument(\"--base-model\", required=True)\n    parser.add_argument(\"--adapter\", required=True)\n    parser.add_argument(\"--previous-metrics\", default=\"\")\n    parser.add_argument(\"--output\", required=True)\n    parser.add_argument(\"--seed\", type=int, default=42)\n    parser.add_argument(\"--max-validation-rows\", type=int, default=1200)\n    parser.add_argument(\"--max-length\", type=int, default=384)\n    parser.add_argument(\"--batch-size\", type=int, default=8)\n    parser.add_argument(\"--swap-rows\", type=int, default=128)\n    main(parser.parse_args())\n", encoding='utf-8')
(source_dir / 'context_length_ablation.py').write_text("\"\"\"Inference-only context-length ablation for the SAVED Qwen-LoRA adapter.\n\nThis module NEVER trains model weights and NEVER submits to Kaggle. It evaluates\none previously trained private adapter on the exact same 1,200-row pilot\nvalidation subset while changing only the tokenizer max_length. Aggregate-only\nJSON is written; no prompts, IDs, per-row predictions, or model artifacts.\n\"\"\"\nimport argparse\nimport json\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom src.saved_adapter_inference import (\n    aggregate_predictions,\n    character_cap_report,\n    exact_pilot_validation,\n    predict_in_batches,\n    token_truncation_report,\n    validated_swap_rows,\n)\nfrom src.baseline import flip_pairs\n\n\nDEFAULT_CONTEXTS = (384, 768, 1024)\nDEFAULT_BATCHES = {384: 8, 768: 4, 1024: 2}\n\n\ndef validate_contexts(contexts):\n    values = tuple(int(x) for x in contexts)\n    if not values or any(x <= 0 for x in values):\n        raise ValueError(\"Context lengths must be positive integers\")\n    if len(set(values)) != len(values):\n        raise ValueError(\"Context lengths must be unique\")\n    return values\n\n\ndef delta_from_baseline(results, baseline_context):\n    base = results[str(baseline_context)]\n    out = {}\n    for key, record in results.items():\n        if int(key) == baseline_context:\n            continue\n        out[key] = {\n            \"delta_log_loss_vs_baseline\":\n                record[\"multiclass_log_loss\"] - base[\"multiclass_log_loss\"],\n            \"delta_accuracy_vs_baseline\":\n                record[\"accuracy\"] - base[\"accuracy\"],\n            \"delta_ece_vs_baseline\":\n                record[\"expected_calibration_error_10_bin\"] -\n                base[\"expected_calibration_error_10_bin\"],\n            \"delta_swap_mean_abs_disagreement_vs_baseline\":\n                record[\"swap_diagnostic\"][\"mean_absolute_probability_disagreement\"] -\n                base[\"swap_diagnostic\"][\"mean_absolute_probability_disagreement\"],\n            \"delta_fraction_swap_changed_winner_vs_baseline\":\n                record[\"swap_diagnostic\"][\"fraction_rows_with_changed_predicted_winner\"] -\n                base[\"swap_diagnostic\"][\"fraction_rows_with_changed_predicted_winner\"],\n            \"delta_truncation_fraction_vs_baseline\":\n                record[\"rendered_token_truncation\"][\"fraction_rendered_sequence_exceeds_token_limit\"] -\n                base[\"rendered_token_truncation\"][\"fraction_rendered_sequence_exceeds_token_limit\"],\n        }\n    return out\n\n\ndef run_ablation(args):\n    # Lazy GPU imports preserve CPU-only synthetic CI.\n    import torch\n    from peft import PeftModel\n    from safetensors import safe_open\n    from transformers import AutoModelForSequenceClassification, AutoTokenizer\n\n    contexts = validate_contexts(args.contexts)\n    base_dir = Path(args.base_model)\n    adapter_dir = Path(args.adapter)\n    if not (base_dir / \"config.json\").is_file():\n        raise FileNotFoundError(\"Missing local Qwen base config\")\n    if not (adapter_dir / \"adapter_config.json\").is_file():\n        raise FileNotFoundError(\"Missing existing adapter_config.json\")\n    if not (adapter_dir / \"adapter_model.safetensors\").is_file():\n        raise FileNotFoundError(\"Missing existing adapter_model.safetensors\")\n    with safe_open(\n        adapter_dir / \"adapter_model.safetensors\", framework=\"pt\", device=\"cpu\"\n    ) as handle:\n        keys = list(handle.keys())\n    head_keys = [k for k in keys if \"score\" in k and k.endswith(\".weight\")]\n    if not head_keys:\n        raise ValueError(\"Saved adapter does not contain classifier score weights\")\n    if not torch.cuda.is_available():\n        raise RuntimeError(\"Context ablation requires Kaggle CUDA for inference only\")\n\n    raw = pd.read_csv(args.train)\n    x_val, y_val = exact_pilot_validation(\n        raw, seed=args.seed, max_validation_rows=args.max_validation_rows\n    )\n    raw_val = raw.loc[x_val.index]\n    n_swap = validated_swap_rows(args.swap_rows, len(x_val))\n\n    tokenizer = AutoTokenizer.from_pretrained(\n        base_dir, local_files_only=True, trust_remote_code=False\n    )\n    if tokenizer.pad_token_id is None:\n        if tokenizer.eos_token is None:\n            raise ValueError(\"Tokenizer has neither pad nor EOS token\")\n        tokenizer.pad_token = tokenizer.eos_token\n    model_max = getattr(tokenizer, \"model_max_length\", None)\n    finite_model_max = (\n        int(model_max) if isinstance(model_max, (int, np.integer))\n        and model_max < 10**9 else None\n    )\n    if finite_model_max and max(contexts) > finite_model_max:\n        raise ValueError(\n            f\"Requested context {max(contexts)} exceeds tokenizer model_max_length \"\n            f\"{finite_model_max}\"\n        )\n\n    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\n    base = AutoModelForSequenceClassification.from_pretrained(\n        base_dir, num_labels=3, torch_dtype=dtype,\n        local_files_only=True, trust_remote_code=False,\n    )\n    base.config.pad_token_id = tokenizer.pad_token_id\n    base.config.use_cache = False\n    model = PeftModel.from_pretrained(\n        base, adapter_dir, is_trainable=False, local_files_only=True\n    )\n    model.to(\"cuda:0\")\n    model.eval()\n\n    results = {}\n    for max_length in contexts:\n        batch_size = int(args.batch_size_overrides.get(\n            max_length, DEFAULT_BATCHES.get(max_length, 2)\n        ))\n        started = time.perf_counter()\n        probabilities = predict_in_batches(\n            model, tokenizer, x_val, \"cuda:0\",\n            max_length=max_length, batch_size=batch_size\n        )\n        swapped = predict_in_batches(\n            model, tokenizer, flip_pairs(x_val.iloc[:n_swap]), \"cuda:0\",\n            max_length=max_length, batch_size=batch_size\n        )[:, [1, 0, 2]]\n        metrics = aggregate_predictions(y_val, probabilities)\n        metrics[\"swap_diagnostic\"] = aggregate_predictions(\n            y_val[:n_swap], probabilities[:n_swap],\n            swapped_aligned=swapped\n        )[\"swap_diagnostic\"]\n        metrics[\"rendered_token_truncation\"] = token_truncation_report(\n            tokenizer, x_val, max_length=max_length\n        )\n        metrics[\"inference_batch_size\"] = batch_size\n        metrics[\"wall_clock_seconds_including_swap_probe\"] = (\n            time.perf_counter() - started\n        )\n        results[str(max_length)] = metrics\n        torch.cuda.empty_cache()\n        print(json.dumps({\n            \"context\": max_length,\n            \"log_loss\": metrics[\"multiclass_log_loss\"],\n            \"accuracy\": metrics[\"accuracy\"],\n            \"ece\": metrics[\"expected_calibration_error_10_bin\"],\n            \"swap_changed_winner\":\n                metrics[\"swap_diagnostic\"][\"fraction_rows_with_changed_predicted_winner\"],\n            \"truncation\":\n                metrics[\"rendered_token_truncation\"][\"fraction_rendered_sequence_exceeds_token_limit\"],\n            \"seconds\": metrics[\"wall_clock_seconds_including_swap_probe\"],\n        }, indent=2))\n\n    baseline = int(args.baseline_context)\n    if baseline not in contexts:\n        raise ValueError(\"--baseline-context must be one of --contexts\")\n    best_context = min(\n        contexts, key=lambda c: results[str(c)][\"multiclass_log_loss\"]\n    )\n    report = {\n        \"type\": \"saved_qwen_context_length_ablation\",\n        \"no_new_training\": True,\n        \"no_competition_submission\": True,\n        \"validation_rows\": int(len(y_val)),\n        \"swap_probe_rows\": int(n_swap),\n        \"seed\": int(args.seed),\n        \"contexts_tokens\": list(contexts),\n        \"baseline_context_tokens\": baseline,\n        \"tokenizer_reported_model_max_length\": finite_model_max,\n        \"saved_classifier_weight_tensors\": len(head_keys),\n        \"character_cap_report\": character_cap_report(raw_val),\n        \"results\": results,\n        \"deltas_from_baseline\": delta_from_baseline(results, baseline),\n        \"lowest_observed_validation_log_loss_context\": int(best_context),\n        \"lowest_observed_validation_log_loss\":\n            float(results[str(best_context)][\"multiclass_log_loss\"]),\n        \"interpretation_limits\": [\n            \"The same previously observed validation set is reused across all contexts; \"\n            \"choosing the best context from this table is exploratory tuning, not independent evidence.\",\n            \"Increasing tokenizer max_length cannot recover text already removed by earlier \"\n            \"character caps in normalized_frame/render_pair.\",\n            \"The adapter was trained with 384-token inputs. Evaluating it at longer context \"\n            \"lengths changes inference exposure without retraining positional usage.\",\n            \"Any improvement or deterioration cannot be attributed solely to truncation without \"\n            \"a prospectively isolated validation set and balanced-template controls.\",\n            \"A/B swap diagnostics describe model order sensitivity, not human presentation bias.\",\n        ],\n    }\n    Path(args.output).parent.mkdir(parents=True, exist_ok=True)\n    Path(args.output).write_text(\n        json.dumps(report, ensure_ascii=False, indent=2) + \"\\n\",\n        encoding=\"utf-8\",\n    )\n    return report\n\n\nif __name__ == \"__main__\":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\"--train\", required=True)\n    parser.add_argument(\"--base-model\", required=True)\n    parser.add_argument(\"--adapter\", required=True)\n    parser.add_argument(\"--output\", required=True)\n    parser.add_argument(\"--seed\", type=int, default=42)\n    parser.add_argument(\"--max-validation-rows\", type=int, default=1200)\n    parser.add_argument(\"--swap-rows\", type=int, default=128)\n    parser.add_argument(\"--contexts\", type=int, nargs=\"+\", default=list(DEFAULT_CONTEXTS))\n    parser.add_argument(\"--baseline-context\", type=int, default=384)\n    args = parser.parse_args()\n    args.batch_size_overrides = DEFAULT_BATCHES\n    run_ablation(args)\n", encoding='utf-8')
sys.path.insert(0, '/kaggle/working')
from src.context_length_ablation import run_ablation


In [ ]:
from argparse import Namespace
params = Namespace(
    train=str(TRAIN), base_model=str(BASE), adapter=str(ADAPTER),
    output='/kaggle/working/qwen_context_ablation.json',
    seed=42, max_validation_rows=1200, swap_rows=128,
    contexts=[384, 768, 1024], baseline_context=384,
    batch_size_overrides={384: 8, 768: 4, 1024: 2},
)
report = run_ablation(params)
print('Lowest observed validation loss context:', report['lowest_observed_validation_log_loss_context'])
print('This is exploratory inference-only tuning, not an independent score or competition submission.')
